In [ ]:
import os
import shutil
import subprocess
import glob
import urllib.request

os.chdir("/content")

print("📦 Установка gdown...")
!pip install -q gdown

# =====================================================================
# 🌐 АВТОМАТИЧЕСКАЯ ЗАГРУЗКА ЧАСТЕЙ АРХИВА С GOOGLE DRIVE
# =====================================================================
parts_info = [
    ("1WmL2k-CORm9SkzX6AgN9cOAYuHBwqtf_", "/content/AIGurlDeps.tar.gz.00"),
    ("1sH0av4ikJz2U54kvkzHMkWHQxW-wTZD1", "/content/AIGurlDeps.tar.gz.01"),
    ("1CmTVYpiWUbq6kFyYtdMwesAxSJB2euax", "/content/AIGurlDeps.tar.gz.02")
]

for file_id, dest_path in parts_info:
    if not os.path.exists(dest_path):
        print(f"📥 Скачивание части архива ({os.path.basename(dest_path)})...")
        !gdown --id {file_id} -O {dest_path}

print("📦 Установка системных зависимостей (sox, libsox-dev, pigz, unzip)...")
!apt-get update -y && apt-get install -y sox libsox-dev pigz unzip -qq

print("🚀 Установка Miniconda (Python 3.10) с принятием TOS...")
!export CONDA_PLUGINS_AUTO_ACCEPT_TOS=true && \
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh && \
bash miniconda.sh -b -p /content/miniconda && \
rm miniconda.sh

print("🚀 Клонирование CosyVoice рекурсивно...")
if not os.path.exists("/content/CosyVoice"):
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git

# =====================================================================
# 🌐 СКЛЕЙКА И РАСПАКОВКА ЧАСТЕЙ АРХИВА
# =====================================================================
tar_parts = sorted(glob.glob("/content/AIGurlDeps.tar.gz.*"))
tar_path = "/content/AIGurlDeps.tar.gz"

if tar_parts and not os.path.exists(tar_path):
    print(f"📦 Найдены части архива ({len(tar_parts)} шт.). Склеиваем в {tar_path}...")
    with open(tar_path, "wb") as wfd:
        for f in tar_parts:
            with open(f, "rb") as rfd:
                shutil.copyfileobj(rfd, wfd)
    print("✅ Склейка архива завершена!")

os.chdir("/content/CosyVoice")
!git submodule update --init --recursive

if os.path.exists(tar_path):
    print("📦 Найден полный архив зависимостей AIGurlDeps.tar.gz. Распаковываем...")
    !apt-get update -y && apt-get install -y pigz -qq
    !tar -I pigz -xf AIGurlDeps.tar.gz -C /content
    print("✅ Окружение и CosyVoice успешно восстановлены из архива!")
else:
    print("🛠️ Удаление openai-whisper из requirements.txt для предотвращения сбоя сборки...")
    !sed -i '/openai-whisper/d' requirements.txt

    print("⚙️ Создание conda-окружения 'cosyvoice' и установка requirements.txt...")
    install_cmds = (
        "export PATH=/content/miniconda/bin:$PATH && "
        "conda create -n cosyvoice -y python=3.10 && "
        "conda run -n cosyvoice pip install -r requirements.txt -i https://mirrors.aliyun.com/pypi/simple/ --trusted-host=mirrors.aliyun.com && "
        "conda run -n cosyvoice pip install openai-whisper --prefer-binary"
    )
    subprocess.run(["bash", "-c", install_cmds], check=True)

    print("📥 Скачивание претренированных моделей и ttsfrd (ModelScope SDK)...")
    download_models_cmd = (
        "export PATH=/content/miniconda/bin:$PATH && "
        "conda run -n cosyvoice python -c \""
        "from modelscope import snapshot_download; "
        "import os; "
        "os.makedirs('pretrained_models', exist_ok=True); "
        "snapshot_download('FunAudioLLM/Fun-CosyVoice3-0.5B-2512', local_dir='pretrained_models/Fun-CosyVoice3-0.5B'); "
        "snapshot_download('iic/CosyVoice2-0.5B', local_dir='pretrained_models/CosyVoice2-0.5B'); "
        "snapshot_download('iic/CosyVoice-300M', local_dir='pretrained_models/CosyVoice-300M'); "
        "snapshot_download('iic/CosyVoice-300M-SFT', local_dir='pretrained_models/CosyVoice-300M-SFT'); "
        "snapshot_download('iic/CosyVoice-300M-Instruct', local_dir='pretrained_models/CosyVoice-300M-Instruct'); "
        "snapshot_download('iic/CosyVoice-ttsfrd', local_dir='pretrained_models/CosyVoice-ttsfrd')"
        "\""
    )
    subprocess.run(["bash", "-c", download_models_cmd], check=True)

    print("📦 Распаковка и установка ttsfrd для качественной нормализации текста...")
    ttsfrd_install_cmd = (
        "export PATH=/content/miniconda/bin:$PATH && "
        "cd pretrained_models/CosyVoice-ttsfrd/ && "
        "if [ -f resource.zip ]; then unzip -o resource.zip -d .; fi && "
        "conda run -n cosyvoice pip install ttsfrd_dependency-0.1-py3-none-any.whl && "
        "conda run -n cosyvoice pip install ttsfrd-0.4.2-cp310-cp310-linux_x86_64.whl"
    )
    subprocess.run(["bash", "-c", ttsfrd_install_cmd], check=True)

    other_deps_install_cmd = (
        "export PATH=/content/miniconda/bin:$PATH && "
        "conda run -n cosyvoice pip install --upgrade pip wheel 'setuptools==69.5.1' && "
        "conda run -n cosyvoice pip install matcha-tts"
    )
    subprocess.run(["bash", "-c", other_deps_install_cmd], check=True)

    print("✅ Официальная установка окружения 'cosyvoice', моделей и ttsfrd успешно завершена!")

# Переходим в рабочую директорию CosyVoice
os.chdir("/content/CosyVoice")

# =====================================================================
# 📥 СКАЧИВАНИЕ server.py ИЗ РЕПОЗИТОРИЯ GITHUB
# =====================================================================
SERVER_URL = "https://raw.githubusercontent.com/ChronoMonochrome/AIGurl/refs/heads/master/server.py"
print(f"📥 Скачивание server.py из репозитория ({SERVER_URL})...")

try:
    urllib.request.urlretrieve(SERVER_URL, "server.py")
    print("✅ server.py успешно скачан!")
except Exception as e:
    print(f"⚠️ Ошибка при скачивании server.py: {e}")
    raise e

print("🚀 Запуск сервера через Python 3.10 с передачей переменных окружения...")

# Подготавливаем окружение с захардкоженными токенами
env = os.environ.copy()
env["NGROK_TOKEN"] = ""
env["STATIC_URL"] = ""

process = subprocess.Popen(
    ["/content/miniconda/envs/cosyvoice/bin/python", "server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

for line in process.stdout:
    print(line, end="")

process.wait()
if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, "server.py")

In [ ]:
from google.colab import drive
import glob
import shutil
import os

# 1. Монтируем Google Drive
print("🔗 Подключение Google Drive...")
drive.mount('/content/drive')

# 2. Создаем папку для бэкапа на Драйве
backup_dir = '/content/drive/MyDrive/AIGurl_Backup'
os.makedirs(backup_dir, exist_ok=True)

# 3. Находим все части архива (AIGurlDeps.tar.gz.00, .01 и т.д.)
archive_parts = sorted(glob.glob('/content/AIGurlDeps.tar.gz.*'))

if not archive_parts:
    print("⚠️ Части архива не найдены в директории /content/")
else:
    print(10 * "-" + f"\nНайдено частей для копирования: {len(archive_parts)}")
    for part in archive_parts:
        filename = os.path.basename(part)
        dest_path = os.path.join(backup_dir, filename)

        print(f"📦 Копирование {filename} на Google Drive...")
        shutil.copy(part, dest_path)
        print(f"✅ Успешно скопировано в: {dest_path}")

    print("\n🎉 Все части архива надежно сохранены на вашем Google Drive!")
    print(f"📁 Путь на диске: Драйв -> AIGurl_Backup")